# Chapter 17 — From Sequences to Attention (Bridge Chapter)

> Bridge from RNNs (Ch.8) to Transformers (Ch.18). The entire point of this notebook: convince yourself that **attention is just a soft dictionary lookup** built out of two primitives you already know — dot product and softmax.

## 1 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (8, 4)

## 2 · Building Block 1 — Dot Product as Similarity

Three unit vectors. The dot product equals $\cos\theta$ for unit vectors, so it is literally the cosine of the angle between them.

In [ ]:
def unit(v):
    return v / np.linalg.norm(v)

a = unit(np.array([1.0, 0.0]))
b = unit(np.array([0.9, 0.1]))    # nearly parallel to a
c = unit(np.array([0.0, 1.0]))    # orthogonal to a
d = unit(np.array([-1.0, 0.0]))   # opposite of a

for name, v in [('b (similar)', b), ('c (orthogonal)', c), ('d (opposite)', d)]:
    print(f'a · {name:20s} = {float(a @ v):+.3f}')

## 3 · Building Block 2 — Softmax with Temperature

Softmax turns a vector of scores into a probability distribution. The temperature $\tau$ controls how sharp the distribution is. Low $\tau$ → one-hot (hard argmax). High $\tau$ → uniform (no preference).

In [ ]:
def softmax(x, tau=1.0, axis=-1):
    x = x / tau
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

scores = np.array([2.0, 1.0, 0.1])
for tau in [0.1, 0.5, 1.0, 2.0, 10.0]:
    p = softmax(scores, tau=tau)
    print(f'tau={tau:>4}  softmax = {np.round(p, 3)}')

In [ ]:
# Visualise the temperature sweep on one shared score vector.
taus = [0.1, 0.5, 1.0, 2.0, 10.0]
scores = np.array([2.0, 1.0, 0.1, -0.5, -1.2])

fig, axes = plt.subplots(1, len(taus), figsize=(14, 3), sharey=True)
for ax, tau in zip(axes, taus):
    p = softmax(scores, tau=tau)
    ax.bar(range(len(p)), p)
    ax.set_title(f'tau = {tau}')
    ax.set_ylim(0, 1)
    ax.set_xticks(range(len(p)))
axes[0].set_ylabel('softmax weight')
fig.suptitle('Softmax temperature sweep — same scores, different tau')
plt.tight_layout()
plt.show()

**Read this plot:** low $\tau$ collapses everything onto the winning score (gradients die — this is why dot products need to be scaled in Ch.18). High $\tau$ flattens the distribution until the model pays equal attention to everything (no information is extracted).

## 4 · Hard Lookup vs Soft Lookup

A Python `dict` matches exactly one key. Attention compares the query to *every* key and returns a **blend** of all values.

In [ ]:
# HARD lookup — the baseline we already understand
prices = {'apple': 0.80, 'banana': 0.30, 'cherry': 2.50}
print('HARD lookup: prices["apple"] =', prices['apple'])

In [ ]:
# SOFT lookup — attention from first principles

def soft_lookup(q, keys, values, tau=1.0):
    scores  = keys @ q / tau            # (n_keys,)
    weights = softmax(scores)           # (n_keys,)
    output  = weights @ values          # blend
    return output, weights

# Fruit embeddings — pretend someone trained these for us
fruit_names = ['apple', 'banana', 'cherry']
fruit_emb   = np.array([
    [0.9, 0.1, 0.0],   # apple
    [0.1, 0.9, 0.0],   # banana
    [0.0, 0.0, 1.0],   # cherry
])
prices_vec = np.array([0.80, 0.30, 2.50]).reshape(-1, 1)

# Query: something 'apple-ish' but not exactly apple
q = np.array([0.7, 0.3, 0.0])
out, w = soft_lookup(q, fruit_emb, prices_vec)

print('query:', q)
for name, weight in zip(fruit_names, w):
    print(f'  attention to {name:6s} = {weight:.3f}')
print(f'soft-lookup price = {float(out):.3f}')

The query is not exactly `apple` — it is *apple-ish*. The soft lookup returns a **blend** of all three prices, weighted by how similar the query is to each key. That is the entire mechanism of attention. Ch.18's $\text{softmax}(QK^\top/\sqrt{d_k})V$ is this same equation written in matrix form.

## 5 · Self-Attention on California Housing Features

Now use the running example. Treat each of the 8 features of a single district as a token; compute the $8 \times 8$ attention matrix.

In [ ]:
feature_names = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                 'Population', 'AveOccup', 'Latitude', 'Longitude']
T, d = 8, 4
X = rng.normal(size=(T, d))         # in Ch.18 these become learned embeddings

# Simplest possible attention: Q = K = V = X (a true bridge — no projections).
scores = X @ X.T / np.sqrt(d)       # (T, T)  — /sqrt(d) previews Ch.18 scaling
A      = softmax(scores, axis=-1)   # (T, T)  — each row sums to 1
out    = A @ X                       # (T, d)  — context-aware outputs

print('attention matrix shape:', A.shape)
print('each row sums to      :', np.round(A.sum(axis=1), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(A, cmap='viridis', vmin=0, vmax=A.max())
ax.set_xticks(range(T)); ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.set_yticks(range(T)); ax.set_yticklabels(feature_names)
ax.set_xlabel('key (attended-to)')
ax.set_ylabel('query (attending)')
ax.set_title('Self-attention matrix — each row is a probability distribution')
for i in range(T):
    for j in range(T):
        ax.text(j, i, f'{A[i, j]:.2f}', ha='center', va='center',
                color='white' if A[i, j] < A.max() * 0.6 else 'black', fontsize=8)
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Read the matrix:** row $i$ tells you how much feature $i$ (as a query) pays attention to each other feature. In a trained transformer these weights would be meaningful — "income attends to location". Here they are noise, because nothing has been trained. The *mechanism*, though, is identical to Ch.18.

## 6 · Attention Is Permutation-Equivariant

Shuffle the input. The output shuffles identically. No notion of order is learned — which is exactly why Ch.18 needs positional encoding.

In [ ]:
def self_attention(X):
    d = X.shape[-1]
    A = softmax(X @ X.T / np.sqrt(d), axis=-1)
    return A @ X

perm = rng.permutation(T)
X_shuffled = X[perm]

out_original = self_attention(X)
out_shuffled = self_attention(X_shuffled)

# If attention is permutation-equivariant, un-shuffling the shuffled output
# should give exactly the same matrix as the original output.
inverse_perm = np.argsort(perm)
recovered    = out_shuffled[inverse_perm]

print('permutation            :', perm)
print('max |original - recovered| =', np.max(np.abs(out_original - recovered)))
print('→ essentially zero — attention is permutation-equivariant.')

## 7 · Exercises

1. **Temperature on the housing matrix.** Re-run Section 5 with `tau = 0.01` and `tau = 100`. What happens to the attention matrix? Which case makes the output informative?
2. **Cross-attention.** In Section 4, replace `prices_vec` with a richer value matrix (e.g. `[price, shelf_life, ripeness]`). What does the soft-lookup output now represent?
3. **Break permutation equivariance.** Add a simple positional encoding (e.g. `X + np.arange(T)[:, None]`) and re-run Section 6. Does the output still shuffle cleanly? Why not?

## Bridge to Ch.18

Everything you saw here — dot product similarity, softmax over keys, weighted sum of values, $T \times T$ attention matrix, permutation equivariance — reappears in Ch.18. The only new ideas will be: (a) learned $W_Q, W_K, W_V$ projections instead of reusing $X$, (b) multiple heads in parallel, (c) positional encoding to fix equivariance, and (d) residuals + LayerNorm + FFN wrapping the whole thing into an encoder block.